# 第一份完整盘口快照：全部原始行

源文件：2026-10-01/01/WIFUSDC_orderbook.parquet。按文件起始连续snapshot组提取，不将后续快照或update混入。每行是一档；以下完整表不省略任何行或字段，保持源文件顺序，未按价格排序。这里的完整指源文件这份快照的全部记录，不保证交易所无限远端价位都被覆盖。

数据输出保存在项目data目录，不会被Git跟踪。

In [ ]:
from pathlib import Path
import pyarrow.parquet as pq
import pyarrow as pa
import pandas as pd
from IPython.display import display, HTML
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'data/cryptohftdata/raw').is_dir())
source = root/'data/cryptohftdata/raw/binance_futures/2026-10-01/01/WIFUSDC_orderbook.parquet'
table = pq.read_table(source)
rows = table.to_pylist()
first = rows[0]
assert first['event_type'] == 'snapshot'
fields = ['event_type','received_time','event_time','last_update_id','final_update_id']
key = tuple(first[f] for f in fields)
snapshot = []
for row in rows:
    if tuple(row[f] for f in fields) != key:
        break
    snapshot.append(row)
frame = pa.Table.from_pylist(snapshot, schema=table.schema).to_pandas(types_mapper=pd.ArrowDtype)
print('完整快照总行数：', len(frame))
print('买盘行数：', sum(frame.side == 'bid'), '；卖盘行数：', sum(frame.side == 'ask'))
print('last_update_id：', first['last_update_id'])
# to_html显式不限制行数；用滚动容器展示完整表，不使用head或省略号。
html = frame.to_html(index=True, max_rows=None, max_cols=None, border=0)
display(HTML('<div style="overflow-x:auto">'+html+'</div>'))
output = root/'data/cryptohftdata/reports/WIFUSDC第一份完整快照.html'
output.write_text('<!doctype html><meta charset="utf-8"><title>WIFUSDC第一份完整快照</title><style>body{font-family:system-ui;padding:24px}table{border-collapse:collapse;font-size:13px}td,th{border:1px solid #ddd;padding:6px;white-space:nowrap}thead{position:sticky;top:0;background:#eee}</style><h1>WIFUSDC第一份完整快照</h1><p>'+str(len(frame))+'行，所有原始字段，源文件顺序。</p>'+html)
print('完整表已另存：', output)
